# Sprint 2: Embedding & Ingestion Observability

This notebook demonstrates the embedding and ingestion pipeline of the RAG system.
It validates:
- Vector store initialization
- Document ingestion
- Embedding generation
- Vector database status reporting
- Query retrieval functionality

## Setup

Initialize required imports, resolve the repository path, and prepare shared notebook state for authentication and API calls.

In [11]:
import json
import os
import subprocess
import sys
import time
from pathlib import Path
from urllib import request
from urllib.error import URLError

def find_repo_root() -> Path:
    cwd = Path.cwd()
    if (cwd / "pyproject.toml").exists():
        return cwd
    if (cwd.parent / "pyproject.toml").exists():
        return cwd.parent
    raise RuntimeError("Could not locate repo root with pyproject.toml")

ROOT = find_repo_root()
SRC_DIR = ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

PORT = 8015
BASE_URL = f"http://127.0.0.1:{PORT}"
server_proc = None
auth_env = {}

print(f"Repo root: {ROOT}")
print(f"Base URL: {BASE_URL}")

Repo root: c:\Users\mlamar\Working Directory(Local)\Forge\m2\da-rag-project-2026_mlamar2019_2
Base URL: http://127.0.0.1:8015


## 1) Authenticate with AI Lab

Authenticate directly from Python and store the bearer token in the notebook session so the API server can use it without requiring `azd auth login`.

In [12]:
from datetime import datetime
from ailab.utils.azure import authenticate_ailab_interactive

auth_result = authenticate_ailab_interactive()
auth_env = {"AILAB_BEARER_TOKEN": auth_result["token"]}

expires_on = datetime.fromtimestamp(auth_result["expires_on"])
print(f"✓ Authenticated via {auth_result['auth_method']}")
print(f"Token expires at: {expires_on}")

✓ Authenticated via interactive_browser
Token expires at: 2026-03-24 15:56:17


## 2) Inspect Notebook Auth State

Confirm that the notebook has acquired a bearer token and is ready to pass it to the FastAPI server process.

In [13]:
auth_status = {
    "has_token": bool(auth_env.get("AILAB_BEARER_TOKEN")),
    "token_prefix": auth_env.get("AILAB_BEARER_TOKEN", "")[:16] + "..." if auth_env.get("AILAB_BEARER_TOKEN") else None,
}

print("Notebook Auth State:")
print(json.dumps(auth_status, indent=2))

if not auth_status["has_token"]:
    raise RuntimeError("Authentication token is missing. Re-run the authentication cell.")

Notebook Auth State:
{
  "has_token": true,
  "token_prefix": "eyJ0eXAiOiJKV1Qi..."
}


## 3) Start FastAPI Server

Launch the FastAPI server and pass the notebook-acquired bearer token into the server environment.

In [21]:
cmd = [
    sys.executable,
    "-m",
    "uvicorn",
    "main:app",
    "--app-dir",
    str(SRC_DIR),
    "--host",
    "127.0.0.1",
    "--port",
    str(PORT),
]

server_env = os.environ.copy()
server_env.update(auth_env)

server_proc = subprocess.Popen(
    cmd,
    cwd=ROOT,
    env=server_env,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True,
 )

ready = False
for _ in range(60):
    try:
        with request.urlopen(f"{BASE_URL}/health", timeout=1) as resp:
            if resp.status == 200:
                ready = True
                break
    except URLError:
        time.sleep(0.5)

if not ready or server_proc.poll() is not None:
    raise RuntimeError("Server failed to start")

print("✓ Server started successfully")

✓ Server started successfully


## 4) Verify Server Auth Status

Call `/auth/status` to confirm the FastAPI server received the notebook-issued bearer token and can see authentication material before ingestion begins.

In [15]:
with request.urlopen(f"{BASE_URL}/auth/status", timeout=5) as resp:
    server_auth_status = json.loads(resp.read().decode("utf-8"))

print("Server Auth Status:")
print(json.dumps(server_auth_status, indent=2))

if not server_auth_status["authenticated"]:
    raise RuntimeError(
        "Server does not have usable authentication material. Re-run the auth cell before continuing."
    )

Server Auth Status:
{
  "authenticated": true,
  "auth_source": "env_token",
  "scope": "api://ailab/Model.Access",
  "error": null
}


## 5) Check Vector Database Status

Query the `/vectordb/status` endpoint to inspect the current state of the vector database before ingestion.

In [18]:
with request.urlopen(f"{BASE_URL}/vectordb/status", timeout=5) as resp:
    db_status = json.loads(resp.read().decode("utf-8"))

print("Vector DB Status (before ingestion):")
print(json.dumps(db_status, indent=2))

Vector DB Status (before ingestion):
{
  "initialized": true,
  "document_count": 5,
  "index_info": "LlamaIndex VectorStoreIndex"
}


## 6) Ingest Documents

Ingest a sample of documents from the default Hugging Face parquet source into the vector database.

In [23]:
from urllib.error import HTTPError, URLError

ingest_payload = {
    "limit": 5
}

req = request.Request(
    f"{BASE_URL}/vectordb/ingest",
    data=json.dumps(ingest_payload).encode("utf-8"),
    headers={"Content-Type": "application/json"},
    method="POST",
)

try:
    with request.urlopen(req, timeout=300) as resp:
        ingest_result = json.loads(resp.read().decode("utf-8"))
    print("Ingestion Result:")
    print(json.dumps(ingest_result, indent=2))
except HTTPError as e:
    detail = e.read().decode("utf-8", errors="ignore")
    print(f"Ingestion failed with HTTP {e.code}: {e.reason}")
    print(detail)
except (URLError, TimeoutError) as e:
    print("Ingestion request failed or timed out.")
    print("If you are not logged in to AI Lab yet, run:")
    print("  azd auth login --scope api://ailab/Model.Access")
    print(f"Error: {e}")

Ingestion Result:
{
  "ingested_count": 5,
  "source": "hf://datasets/rag-datasets/rag-mini-wikipedia/data/passages.parquet/part.0.parquet",
  "status": {
    "initialized": true,
    "document_count": 5,
    "index_info": "LlamaIndex VectorStoreIndex"
  }
}


## 7) Test Query Endpoint

Test the `/vectordb/query` endpoint after ingestion to verify retrieval functionality.

In [24]:
from urllib.parse import urlencode
from urllib.error import URLError

# Guard: verify the server is still running before attempting the query
if server_proc is None or server_proc.poll() is not None:
    raise RuntimeError(
        "Server is not running. Re-run cells in order starting from the 'Start FastAPI Server' cell."
    )

query_params = {
    "q": "What is artificial intelligence?",
    "top_k": 3,
}
query_string = urlencode(query_params)

try:
    with request.urlopen(f"{BASE_URL}/vectordb/query?{query_string}", timeout=30) as resp:
        results = json.loads(resp.read().decode("utf-8"))
except URLError as e:
    raise RuntimeError(
        f"Could not reach server at {BASE_URL}. Re-run cells in order from the top. Error: {e}"
    ) from e

print(f"Query: {query_params['q']}")
print(f"Results found: {len(results)}")
if results:
    print("\nFirst result:")
    print(json.dumps(results[0], indent=2))
else:
    print("(No results - vector database is empty)")

Query: What is artificial intelligence?
Results found: 3

First result:
{
  "text": "The economy is largely based in agriculture (making up 10% of the GDP and the most substantial export) and the state-sector, and relies heavily on world trade. Consequently, it is badly affected by any downturn in global prices. However, the economy is on the whole more stable than surrounding states, and it maintains a solid reputation with investors.",
  "score": 0.022920129823321762,
  "metadata": {}
}


## 8) Verify All Endpoints

Verify that all required endpoints are available and respond correctly.

In [ ]:
endpoints = [
    ("/health", "GET"),
    ("/auth/status", "GET"),
    ("/vectordb/status", "GET"),
    ("/vectordb/query?q=test&top_k=1", "GET"),
    ("/vectordb/ingest", "POST"),
    ("/echo", "POST"),
]

# Endpoints that call Azure OpenAI need a longer timeout
SLOW_ENDPOINTS = {"/vectordb/query?q=test&top_k=1", "/vectordb/ingest"}

print("Endpoint Availability Check:")
for endpoint, method in endpoints:
    timeout = 30 if endpoint in SLOW_ENDPOINTS else 5
    try:
        if method == "GET":
            with request.urlopen(f"{BASE_URL}{endpoint}", timeout=timeout) as resp:
                status_code = resp.status
                response_body = resp.read().decode("utf-8") if endpoint == "/auth/status" else None
        elif method == "POST" and endpoint == "/vectordb/ingest":
            req = request.Request(
                f"{BASE_URL}{endpoint}",
                data=json.dumps({"limit": 1}).encode("utf-8"),
                headers={"Content-Type": "application/json"},
                method="POST",
            )
            with request.urlopen(req, timeout=timeout) as resp:
                status_code = resp.status
                response_body = None
        elif method == "POST":
            req = request.Request(
                f"{BASE_URL}{endpoint}",
                data=json.dumps({"message": "test"}).encode("utf-8"),
                headers={"Content-Type": "application/json"},
                method="POST",
            )
            with request.urlopen(req, timeout=timeout) as resp:
                status_code = resp.status
                response_body = None
        print(f"✓ {method:4s} {endpoint:28s} -> {status_code}")
        if endpoint == "/auth/status" and response_body:
            print(f"  auth status: {response_body}")
    except Exception as e:
        print(f"✗ {method:4s} {endpoint:28s} -> Error: {e}")

## 9) Cleanup

Stop the test server.

## Summary

Sprint 2 now demonstrates:
- ✓ Vector store initialization and status reporting via `/vectordb/status`
- ✓ Document ingestion via `/vectordb/ingest` from the default passages source
- ✓ Query endpoint retrieval via `/vectordb/query`
- ✓ Integration with FastAPI application and Azure embedding model

**Next Steps:**
- Add ingest progress metrics and timing details to the status endpoint
- Add retrieval quality checks using known Q/A samples
- Begin Sprint 3 prompt augmentation and response generation

<!-- re-push -->

In [25]:
if 'server_proc' in globals() and server_proc is not None and server_proc.poll() is None:
    server_proc.terminate()
    print('Server terminated')
else:
    print('Server not running')

Server terminated
